In [27]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import os
import urllib.request
from pathlib import Path

if IN_COLAB:
    drive.mount('/content/drive')
    # Shared project folder (one-time: open folder → Add shortcut to Drive)
    DRIVE_BASE = '/content/drive/.shortcut-targets-by-id/1r4t5aR-3Yal45Gv94bZEj7Pulyx3iLl1/CSC-566-Final_Project_Mel_Spectrograms'
    METADATA_DIR = f'{DRIVE_BASE}/metadata'
    MELSPECS_DIR = f'{DRIVE_BASE}/melspecs'
    LOCAL_MELSPECS = '/content/melspecs'
else:
    PROJECT_ROOT = Path('..').resolve()
    METADATA_DIR = str(PROJECT_ROOT / 'data' / 'metadata')
    MELSPECS_DIR = str(PROJECT_ROOT / 'data' / 'melspecs')
    LOCAL_MELSPECS = MELSPECS_DIR

os.makedirs(LOCAL_MELSPECS, exist_ok=True)

TSV_PATH = f'{METADATA_DIR}/autotagging_moodtheme.tsv'
if not os.path.exists(TSV_PATH):
    fallback_meta = '/content/metadata' if IN_COLAB else METADATA_DIR
    os.makedirs(fallback_meta, exist_ok=True)
    TSV_PATH = f'{fallback_meta}/autotagging_moodtheme.tsv'
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data/autotagging_moodtheme.tsv',
        TSV_PATH,
    )

if not os.path.isdir(MELSPECS_DIR):
    print(f'WARNING: {MELSPECS_DIR} not found.')
    print('Add the shared project folder to Drive: https://drive.google.com/drive/folders/1r4t5aR-3Yal45Gv94bZEj7Pulyx3iLl1')
elif not os.path.exists(TSV_PATH):
    print(f'WARNING: {TSV_PATH} not found.')
else:
    n_mels = len([f for f in os.listdir(MELSPECS_DIR) if f.endswith('.npy')])
    print('Drive mounted and metadata found. Ready to go.')
    print(f'Melspec files on Drive: {n_mels}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted and metadata found. Ready to go.
Melspec files on Drive: 13880


In [28]:
if IN_COLAB:
    !pip install librosa -q

import numpy as np
import librosa.display
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm

In [29]:
import shutil
from concurrent.futures import ThreadPoolExecutor

MAX_TRACKS = 5000

if IN_COLAB:
    files = [f for f in os.listdir(MELSPECS_DIR) if f.endswith('.npy')][:MAX_TRACKS]
    print(f'Copying {len(files)} files...')

    def copy_file(fname):
        src = os.path.join(MELSPECS_DIR, fname)
        dst = os.path.join(LOCAL_MELSPECS, fname)
        if not os.path.exists(dst):
            shutil.copy(src, dst)

    with ThreadPoolExecutor(max_workers=8) as executor:
        list(tqdm(executor.map(copy_file, files), total=len(files), desc='Copying'))

print(f'Done! Files available: {len([f for f in os.listdir(LOCAL_MELSPECS) if f.endswith(".npy")])}')

Copying 5000 files...


Copying: 100%|██████████| 5000/5000 [00:00<00:00, 48608.08it/s]

Done! Files available: 5000


In [30]:
# Check for corrupted .npy files and remove them from both local and Drive
bad_files = []
for fname in os.listdir(LOCAL_MELSPECS):
    if not fname.endswith(".npy"):
        continue
    local_path = os.path.join(LOCAL_MELSPECS, fname)
    try:
        np.load(local_path, mmap_mode="r")
    except Exception as e:
        print(f"Bad: {fname} — {e}")
        bad_files.append(fname)

print(f"{len(bad_files)} corrupted files found.")
for fname in bad_files:
    os.remove(os.path.join(LOCAL_MELSPECS, fname))
    drive_path = os.path.join(MELSPECS_DIR, fname)
    if os.path.exists(drive_path):
        os.remove(drive_path)
    print(f"Deleted: {fname}")

if bad_files:
    print("Re-run the download cell to re-fetch corrupted files.")
else:
    print("All files OK.")


0 corrupted files found.
All files OK.


In [31]:
import pandas as pd

# MOOD/THEME TSV
rows = []
with open(TSV_PATH) as f:
    next(f)  # skip header
    for line in f:
        parts = line.rstrip('\n').split('\t')
        rows.append({
            'TRACK_ID':  parts[0],
            'ARTIST_ID': parts[1],
            'ALBUM_ID':  parts[2],
            'PATH':      parts[3],
            'DURATION':  float(parts[4]),
            'TAGS_MOOD': parts[5:],  # all remaining fields are tags
        })

df = pd.DataFrame(rows)
df.head()

,TRACK_ID,ARTIST_ID,ALBUM_ID,PATH,DURATION,TAGS_MOOD
0,track_0000948,artist_000087,album_000149,48/948.mp3,212.7,[mood/theme---background]
1,track_0000950,artist_000087,album_000149,50/950.mp3,248.0,[mood/theme---background]
2,track_0000951,artist_000087,album_000149,51/951.mp3,199.7,[mood/theme---background]
3,track_0002165,artist_000326,album_000347,65/2165.mp3,229.0,[mood/theme---film]
4,track_0002263,artist_000320,album_000366,63/2263.mp3,494.7,[mood/theme---melancholic]


In [32]:
# Keep mood/theme labels as multi-label targets.
# MTG-Jamendo mood/theme is an auto-tagging task: one track can have several labels.
NUM_TAGS = 20
exploded_tags = df.explode('TAGS_MOOD')
tag_counts = exploded_tags['TAGS_MOOD'].value_counts()
filtered_tag_counts = tag_counts.head(NUM_TAGS)
filtered_tag_counts

top_tags = list(filtered_tag_counts.index)
top_tags_set = set(top_tags)

def keep_top_tags(tags):
    return [tag for tag in tags if tag in top_tags_set]

def apply_label_filter(frame):
    out = frame.copy()
    out['TAGS_MOOD'] = out['TAGS_MOOD'].apply(keep_top_tags)
    out = out[out['TAGS_MOOD'].str.len() > 0].reset_index(drop=True)
    return out

df_filtered = apply_label_filter(df)

label_lengths = df_filtered['TAGS_MOOD'].str.len()
print(f'Filtered dataset: {len(df_filtered)} tracks')
print(f'Multi-label tracks retained: {(label_lengths > 1).sum()} / {len(df_filtered)}')
filtered_tag_counts


Filtered dataset: 13881 tracks
Multi-label tracks retained: 4146 / 13881


,count
TAGS_MOOD,
mood/theme---happy,1657
mood/theme---film,1502
mood/theme---energetic,1357
mood/theme---relaxing,1350
mood/theme---emotional,1271
mood/theme---melodic,1213
mood/theme---dark,1202
mood/theme---epic,982
mood/theme---dream,951


In [33]:
# Multi-hot label encoding.
tag_index_mapping = {tag: i for i, tag in enumerate(top_tags)}
index_tag_mapping = {i: tag for tag, i in tag_index_mapping.items()}

# Encodes all mood/theme tags for a track into one multi-hot vector.
def encode_labels(tags):
    label = np.zeros(len(tag_index_mapping), dtype=np.float32)
    for tag in tags:
        if tag in tag_index_mapping:
            label[tag_index_mapping[tag]] = 1.0
    return label

df_filtered['LABEL_ENCODING'] = df_filtered['TAGS_MOOD'].apply(encode_labels)
df_filtered[['TRACK_ID', 'TAGS_MOOD', 'LABEL_ENCODING']].head()


,TRACK_ID,TAGS_MOOD,LABEL_ENCODING
0,track_0002165,[mood/theme---film],"[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,track_0003346,[mood/theme---melodic],"[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, ..."
2,track_0003347,[mood/theme---melodic],"[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, ..."
3,track_0003348,[mood/theme---melodic],"[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, ..."
4,track_0003349,[mood/theme---melodic],"[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, ..."


In [34]:
from torch.utils.data import Dataset

N_FREQ = 96
# 1344 keeps the encoder/decoder dimensions divisible by the pooling factors.
CROP_LENGTH = 1344


def normalize_mel(mel):
    """Per-track standardization makes reconstruction/classification less sensitive to loudness scale."""
    mel = mel.astype(np.float32)
    return (mel - mel.mean()) / (mel.std() + 1e-6)


def spec_augment(mel, freq_mask_param=8, time_mask_param=40):
    """Zero out random freq/time bands (train-only augmentation)."""
    mel = mel.copy()
    f, t = mel.shape
    if freq_mask_param > 0 and f > freq_mask_param:
        f0 = np.random.randint(0, f - freq_mask_param)
        mel[f0:f0 + freq_mask_param, :] = mel.min()
    if time_mask_param > 0 and t > time_mask_param:
        t0 = np.random.randint(0, t - time_mask_param)
        mel[:, t0:t0 + time_mask_param] = mel.min()
    return mel


class MoodDataset(Dataset):
    def __init__(self, df, melspecs_dir, augment=False, deterministic=False):
        self.df = df.reset_index(drop=True)
        self.melspecs_dir = melspecs_dir
        self.augment = augment
        self.deterministic = deterministic

    def __len__(self):
        return len(self.df)

    def _crop_mel(self, mel):
        t = mel.shape[1]
        if t > CROP_LENGTH:
            if self.deterministic:
                start = (t - CROP_LENGTH) // 2
            else:
                start = np.random.randint(0, t - CROP_LENGTH)
            mel = mel[:, start:start + CROP_LENGTH]
        else:
            mel = np.pad(mel, ((0, 0), (0, CROP_LENGTH - t)))
        return mel

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        track_id = int(row['TRACK_ID'].replace('track_', ''))
        mel = np.load(f"{self.melspecs_dir}/{track_id}.npy")
        if mel.shape[0] != N_FREQ:
            mel = mel.T

        mel = self._crop_mel(mel)
        mel = normalize_mel(mel)
        if self.augment:
            mel = spec_augment(mel)

        mel = torch.tensor(mel, dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(row['LABEL_ENCODING'], dtype=torch.float32)
        return mel, label


In [35]:
from sklearn.model_selection import train_test_split

SPLIT_ID = 0
USE_OFFICIAL_SPLIT = True


def npy_exists(row):
    track_id = int(row['TRACK_ID'].replace('track_', ''))
    return os.path.exists(f"{LOCAL_MELSPECS}/{track_id}.npy")


def get_official_split(split_name):
    split_dir = Path(METADATA_DIR) / 'splits' / f'split-{SPLIT_ID}'
    split_dir.mkdir(parents=True, exist_ok=True)
    filename = f'autotagging_moodtheme-{split_name}.tsv'
    split_path = split_dir / filename
    if not split_path.exists():
        url = f'https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data/splits/split-{SPLIT_ID}/{filename}'
        urllib.request.urlretrieve(url, split_path)

    split_ids = pd.read_csv(split_path, sep='\t', usecols=[0]).iloc[:, 0].astype(str).tolist()
    split_ids = [track_id for track_id in split_ids if track_id != 'TRACK_ID']
    if not split_ids:
        raise ValueError(f'{split_path} did not contain track ids')

    split_order = {track_id: i for i, track_id in enumerate(split_ids)}
    split_df = df_filtered[df_filtered['TRACK_ID'].isin(split_order)].copy()
    if split_df.empty:
        raise ValueError(f'{filename} had no overlap with the filtered top-{NUM_TAGS} metadata')

    split_df['__split_order'] = split_df['TRACK_ID'].map(split_order)
    split_df = split_df.sort_values('__split_order').drop(columns='__split_order').reset_index(drop=True)
    return split_df


def keep_available(frame):
    return frame[frame.apply(npy_exists, axis=1)].reset_index(drop=True)


def make_random_available_split(reason):
    print(f'{reason} Falling back to a random split over locally available mel files.')
    df_available = keep_available(df_filtered)
    if len(df_available) < 3:
        raise ValueError(f'Only {len(df_available)} locally available labeled mel files found.')
    df_train_val, df_test = train_test_split(df_available, test_size=0.2, random_state=42)
    df_train, df_val = train_test_split(df_train_val, test_size=0.2, random_state=42)
    return df_train.reset_index(drop=True), df_val.reset_index(drop=True), df_test.reset_index(drop=True)

try:
    if not USE_OFFICIAL_SPLIT:
        raise RuntimeError('Official split disabled')
    df_train = keep_available(get_official_split('train'))
    df_val = keep_available(get_official_split('validation'))
    df_test = keep_available(get_official_split('test'))

    empty = {name: len(frame) for name, frame in [('train', df_train), ('validation', df_val), ('test', df_test)] if len(frame) == 0}
    if empty:
        raise ValueError(f'Official split has no locally available mel files for: {empty}')
    print(f'Using official MTG-Jamendo split-{SPLIT_ID}.')
except Exception as exc:
    df_train, df_val, df_test = make_random_available_split(f'Could not use official split files ({exc}).')

print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")
print('Train label counts:')
print(pd.Series(np.stack(df_train['LABEL_ENCODING']).sum(axis=0), index=top_tags).sort_values(ascending=False).head(10))


Could not use official split files (autotagging_moodtheme-validation.tsv had no overlap with the filtered top-20 metadata). Falling back to a random split over locally available mel files.
Train: 3200 | Val: 800 | Test: 1000
Train label counts:
mood/theme---happy        425.0
mood/theme---energetic    376.0
mood/theme---film         327.0
mood/theme---relaxing     297.0
mood/theme---emotional    286.0
mood/theme---dark         257.0
mood/theme---melodic      235.0
mood/theme---love         230.0
mood/theme---dream        217.0
mood/theme---epic         206.0
dtype: float32


In [36]:
from torch.utils.data import DataLoader


def make_loaders(df_train, df_val, df_test, melspecs_dir, batch_size=16):
    train_ds = MoodDataset(df_train, melspecs_dir, augment=True, deterministic=False)
    val_ds   = MoodDataset(df_val,   melspecs_dir, augment=False, deterministic=True)
    test_ds  = MoodDataset(df_test,  melspecs_dir, augment=False, deterministic=True)
    loader_kwargs = dict(batch_size=batch_size, num_workers=2, pin_memory=True)

    train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
    val_loader   = DataLoader(val_ds, shuffle=False, **loader_kwargs)
    test_loader  = DataLoader(test_ds, shuffle=False, **loader_kwargs)

    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = make_loaders(
    df_train, df_val, df_test, LOCAL_MELSPECS, batch_size=16
)


In [37]:
import torch.nn as nn
import torch.nn.functional as F


class MoodEmbeddingAutoencoder(nn.Module):
    """Autoencoder-compatible model whose latent space is driven by mood/theme labels."""
    def __init__(self, latent_dim=256, num_tags=20):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 4)),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 4)),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 4)),

            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
        )

        # Attention pooling lets the model decide which time regions support the track-level tags.
        self.attention = nn.Sequential(
            nn.Conv1d(512, 128, kernel_size=1),
            nn.Tanh(),
            nn.Conv1d(128, 1, kernel_size=1),
        )

        self.fc_enc = nn.Sequential(
            nn.Linear(512, latent_dim),
            nn.LayerNorm(latent_dim),
        )

        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, num_tags),
        )

        # Kept as a weak auxiliary decoder; do not let reconstruction dominate the embedding.
        self.fc_dec = nn.Linear(latent_dim, 512 * 12 * 21)
        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=(2, 4)),
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Upsample(scale_factor=(2, 4)),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Upsample(scale_factor=(2, 4)),
            nn.Conv2d(128, 1, kernel_size=3, padding=1),
        )

    def encode_raw(self, x):
        h = self.encoder(x)              # B, 512, 12, 21
        h_time = h.mean(dim=2)           # B, 512, 21
        attn = torch.softmax(self.attention(h_time), dim=-1)
        pooled = (h_time * attn).sum(dim=-1)
        z = self.fc_enc(pooled)
        return z

    def encode(self, x):
        return F.normalize(self.encode_raw(x), p=2, dim=1)

    def decode(self, z):
        x = self.fc_dec(z)
        x = x.view(x.size(0), 512, 12, 21)
        return self.decoder(x)

    def forward(self, x):
        z = self.encode(x)
        logits = self.classifier(z)
        recon = self.decode(z)
        return recon, z, logits


def supervised_multilabel_contrastive_loss(z, labels, temperature=0.1):
    """Pull together tracks sharing at least one mood/theme tag in the normalized embedding space."""
    labels = (labels > 0).float()
    positive_mask = (labels @ labels.T) > 0
    positive_mask.fill_diagonal_(False)

    logits = (z @ z.T) / temperature
    logits = logits - logits.max(dim=1, keepdim=True).values.detach()
    self_mask = torch.eye(z.size(0), device=z.device, dtype=torch.bool)
    logits = logits.masked_fill(self_mask, -1e9)

    log_prob = logits - torch.logsumexp(logits, dim=1, keepdim=True)
    positives_per_anchor = positive_mask.sum(dim=1)
    valid = positives_per_anchor > 0
    if valid.sum() == 0:
        return z.new_tensor(0.0)

    loss = -(log_prob * positive_mask.float()).sum(dim=1) / positives_per_anchor.clamp_min(1)
    return loss[valid].mean()


In [ ]:
import torch
from tqdm import tqdm
from sklearn.metrics import average_precision_score, roc_auc_score


device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

model = MoodEmbeddingAutoencoder(latent_dim=256, num_tags=len(top_tags)).to(device)

train_label_matrix = np.stack(df_train['LABEL_ENCODING'].to_numpy())
pos_counts = train_label_matrix.sum(axis=0)
neg_counts = len(train_label_matrix) - pos_counts
pos_weight = torch.tensor(neg_counts / np.maximum(pos_counts, 1), dtype=torch.float32, device=device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
bce_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
recon_criterion = nn.MSELoss()

CONTRASTIVE_WEIGHT = 0.1
RECON_WEIGHT = 0.01
NUM_EPOCHS = 30
best_val_map = -float('inf')


def evaluate(loader):
    model.eval()
    losses = []
    all_logits = []
    all_labels = []
    all_embeddings = []

    with torch.no_grad():
        for mels, labels in loader:
            mels, labels = mels.to(device), labels.to(device)
            recon, z, logits = model(mels)
            bce_loss = bce_criterion(logits, labels)
            contrastive_loss = supervised_multilabel_contrastive_loss(z, labels)
            recon_loss = recon_criterion(recon, mels)
            loss = bce_loss + CONTRASTIVE_WEIGHT * contrastive_loss + RECON_WEIGHT * recon_loss

            losses.append(loss.item())
            all_logits.append(logits.cpu())
            all_labels.append(labels.cpu())
            all_embeddings.append(z.cpu())

    if not all_labels:
        dataset_size = len(loader.dataset) if hasattr(loader, 'dataset') else 0
        raise ValueError(
            f'Evaluation loader is empty (dataset size: {dataset_size}). ' 
            'Re-run the split and DataLoader cells; if you copied only a subset of mels, use the random fallback split.'
        )

    y_true = torch.cat(all_labels).numpy()
    y_score = torch.sigmoid(torch.cat(all_logits)).numpy()
    embeddings = torch.cat(all_embeddings).numpy()

    ap_scores = []
    auc_scores = []
    for i in range(y_true.shape[1]):
        if len(np.unique(y_true[:, i])) < 2:
            continue
        ap_scores.append(average_precision_score(y_true[:, i], y_score[:, i]))
        auc_scores.append(roc_auc_score(y_true[:, i], y_score[:, i]))

    return {
        'loss': float(np.mean(losses)),
        'macro_pr_auc': float(np.mean(ap_scores)) if ap_scores else np.nan,
        'macro_roc_auc': float(np.mean(auc_scores)) if auc_scores else np.nan,
        'y_true': y_true,
        'y_score': y_score,
        'embeddings': embeddings,
    }


for epoch in range(NUM_EPOCHS):
    model.train()
    train_bce = 0.0
    train_contrastive = 0.0
    train_recon = 0.0

    for mels, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Train"):
        mels, labels = mels.to(device), labels.to(device)

        optimizer.zero_grad()
        recon, z, logits = model(mels)
        bce_loss = bce_criterion(logits, labels)
        contrastive_loss = supervised_multilabel_contrastive_loss(z, labels)
        recon_loss = recon_criterion(recon, mels)
        loss = bce_loss + CONTRASTIVE_WEIGHT * contrastive_loss + RECON_WEIGHT * recon_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        train_bce += bce_loss.item()
        train_contrastive += contrastive_loss.item()
        train_recon += recon_loss.item()

    train_bce /= len(train_loader)
    train_contrastive /= len(train_loader)
    train_recon /= len(train_loader)
    val = evaluate(val_loader)

    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS} | "
        f"Train BCE: {train_bce:.4f} | Contrastive: {train_contrastive:.4f} | Recon: {train_recon:.4f} | "
        f"Val Loss: {val['loss']:.4f} | Val PR-AUC: {val['macro_pr_auc']:.4f} | Val ROC-AUC: {val['macro_roc_auc']:.4f}"
    )

    if val['macro_pr_auc'] > best_val_map:
        best_val_map = val['macro_pr_auc']
        torch.save(model.state_dict(), 'best_mood_embedding_autoencoder.pth')
        print(f"  Best model saved (Val PR-AUC: {best_val_map:.4f})")


Using device: cuda


Epoch 1 Train: 100%|██████████| 200/200 [01:12<00:00,  2.75it/s]


Epoch 1/30 | Train BCE: 1.2890 | Contrastive: 2.6952 | Recon: 2.4151 | Val Loss: 1.5610 | Val PR-AUC: 0.1150 | Val ROC-AUC: 0.6179
  Best model saved (Val PR-AUC: 0.1150)


Epoch 2 Train: 100%|██████████| 200/200 [01:12<00:00,  2.78it/s]


Epoch 2/30 | Train BCE: 1.2819 | Contrastive: 2.6867 | Recon: 2.2936 | Val Loss: 1.5460 | Val PR-AUC: 0.1204 | Val ROC-AUC: 0.6405
  Best model saved (Val PR-AUC: 0.1204)


...

Epoch 11 Train: 100%|██████████| 200/200 [01:11<00:00,  2.81it/s]


Epoch 11/30 | Train BCE: 1.2026 | Contrastive: 2.6845 | Recon: 1.8294 | Val Loss: 1.4701 | Val PR-AUC: 0.1320 | Val ROC-AUC: 0.6580


Epoch 12 Train: 100%|██████████| 200/200 [01:10<00:00,  2.82it/s]


Epoch 12/30 | Train BCE: 1.2007 | Contrastive: 2.6747 | Recon: 1.8041 | Val Loss: 1.5168 | Val PR-AUC: 0.1344 | Val ROC-AUC: 0.6633
  Best model saved (Val PR-AUC: 0.1344)


Epoch 13 Train:  66%|██████▋   | 133/200 [00:47<00:23,  2.90it/s]

In [ ]:
try:
    import umap
except ImportError:
    import sys
    !{sys.executable} -m pip install umap-learn -q
    import umap

import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Load the best tag-supervised embedding model before plotting.
model.load_state_dict(torch.load('best_mood_embedding_autoencoder.pth', map_location=device))
test_eval = evaluate(test_loader)
all_embeddings = test_eval['embeddings']
all_labels = test_eval['y_true']
all_scores = test_eval['y_score']

print(f"Test Macro PR-AUC: {test_eval['macro_pr_auc']:.4f}")
print(f"Test Macro ROC-AUC: {test_eval['macro_roc_auc']:.4f}")
print(f"Embeddings shape: {all_embeddings.shape}")


def primary_tag_names(label_matrix):
    primary_idx = np.argmax(label_matrix, axis=1)
    return np.array([top_tags[i].replace('mood/theme---', '') for i in primary_idx])


def plot_2d(points, labels, title, alpha=0.65):
    unique_tags = sorted(set(labels))
    colors = cm.tab20(np.linspace(0, 1, len(unique_tags)))
    tag_to_color = {tag: colors[i] for i, tag in enumerate(unique_tags)}

    plt.figure(figsize=(12, 8))
    for tag in unique_tags:
        mask = labels == tag
        plt.scatter(points[mask, 0], points[mask, 1], s=12, alpha=alpha, color=tag_to_color[tag], label=tag)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    plt.title(title)
    plt.tight_layout()
    plt.show()


plot_labels = primary_tag_names(all_labels)

# 1) Learned latent space: this is the representation trained for mood/theme similarity.
print('Running UMAP on learned label-supervised embeddings...')
umap_emb = umap.UMAP(n_components=2, n_neighbors=25, min_dist=0.15, metric='cosine', random_state=42).fit_transform(all_embeddings)
plot_2d(umap_emb, plot_labels, 'UMAP of Tag-Supervised Mood/Theme Embeddings')

print('Running PCA on learned label-supervised embeddings...')
pca_emb = PCA(n_components=2, random_state=42).fit_transform(all_embeddings)
plot_2d(pca_emb, plot_labels, 'PCA of Tag-Supervised Mood/Theme Embeddings')

# 2) Semantic score space: PCA/UMAP on predicted tag probabilities often makes label neighborhoods easier to inspect.
print('Running PCA on predicted tag probability space...')
score_pca = PCA(n_components=2, random_state=42).fit_transform(all_scores)
plot_2d(score_pca, plot_labels, 'PCA of Predicted Mood/Theme Probability Space')

# 3) Paper-style feature-space PCA from mel summary statistics.
# The referenced paper visualizes PCA over engineered audio features. With precomputed mels, we can build
# comparable summary features from each mel bin: mean, std, percentiles, and global energy summaries.
def mel_summary_features(frame, max_items=2500):
    feats = []
    labels = []
    subset = frame.reset_index(drop=True).iloc[:max_items]
    for _, row in tqdm(subset.iterrows(), total=len(subset), desc='Extracting mel summary features'):
        track_id = int(row['TRACK_ID'].replace('track_', ''))
        mel = np.load(f"{LOCAL_MELSPECS}/{track_id}.npy")
        if mel.shape[0] != N_FREQ:
            mel = mel.T
        mel = normalize_mel(mel)
        feat = np.concatenate([
            mel.mean(axis=1),
            mel.std(axis=1),
            np.percentile(mel, 10, axis=1),
            np.percentile(mel, 90, axis=1),
            np.array([mel.mean(), mel.std(), mel.min(), mel.max()], dtype=np.float32),
        ])
        feats.append(feat)
        labels.append(row['LABEL_ENCODING'])
    return np.asarray(feats), np.asarray(labels)

feature_matrix, feature_labels = mel_summary_features(df_test)
feature_matrix = StandardScaler().fit_transform(feature_matrix)
feature_pca = PCA(n_components=2, random_state=42).fit_transform(feature_matrix)
plot_2d(feature_pca, primary_tag_names(feature_labels), 'Paper-Style PCA of Engineered Mel Summary Features')
